In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Any, Union
from dataclasses import dataclass, field
from datetime import datetime
import gc
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import (
    T5Config,
    T5ForConditionalGeneration,
    T5Tokenizer,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    TrainerState,
    TrainerControl,
    EarlyStoppingCallback,
    set_seed,
)
from datasets import (
    load_dataset,
    concatenate_datasets,
    Dataset,
    DatasetDict,
)
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
from tqdm.auto import tqdm
# Environment informations
print("=" * 80)
print(" mT5 Pretraining Environment Setup")
print("=" * 80)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("No GPU detected - training will be slow!")
print("=" * 80)

: 

In [ ]:
class PretrainingConfig:
    
    # Dataset paths
    DATASET_FILES = [
        'dataset1.json', 
        'dataset2.json',
        'dataset3.json',
        'dataset4.json',
        'dataset5.json',
    ]
    
    TOKENIZER_PATH = 'google/mt5-small'  
    
    OUTPUT_DIR = './checkpoints/mt5-ja-ne-pretrained'
    LOGGING_DIR = './logs'
    FINAL_MODEL_DIR = './final_model/mt5-ja-ne'
    
    # Model configuration
    MODEL_NAME = 'google/mt5-xl' 
    
    # For training from scratch (set TRAIN_FROM_SCRATCH = True)
    TRAIN_FROM_SCRATCH = False
    VOCAB_SIZE = 250000
    D_MODEL = 512
    D_FF = 2048
    NUM_LAYERS = 8
    NUM_DECODER_LAYERS = 8
    NUM_HEADS = 6
    DROPOUT_RATE = 0.1
    
   # Data configuration
    MAX_LENGTH = 512
    VALIDATION_SPLIT = 0.05  # 5% for validation
    STREAMING = False  # Set True for very large datasets (>RAM)
    NUM_PROC = 4  # Parallel processing workers for tokenization
    
    #Training hyperparameters
    BATCH_SIZE = 8  # Per device batch size
    GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = 8 * 4 = 32
    LEARNING_RATE = 1e-4
    NUM_TRAIN_EPOCHS = 3
    WARMUP_STEPS = 10000
    MAX_STEPS = -1  # -1 means train for full epochs
    
    # Optimization
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 1.0
    ADAM_EPSILON = 1e-8
    ADAM_BETA1 = 0.9
    ADAM_BETA2 = 0.999
    LR_SCHEDULER_TYPE = 'linear'
    
    # Checkpointing & Logging 
    SAVE_STRATEGY = 'steps'
    SAVE_STEPS = 5000
    SAVE_TOTAL_LIMIT = 3  # Keep only 3 best checkpoints
    EVAL_STRATEGY = 'steps'
    EVAL_STEPS = 2500
    LOGGING_STEPS = 100
    
    # Performance Optimization 
    FP16 = True 
    BF16 = False  
    DATALOADER_NUM_WORKERS = 4
    DATALOADER_PIN_MEMORY = True
    GRADIENT_CHECKPOINTING = True  
    OPTIM = 'adamw_torch' 
    
    # T5 Pretraining Specific
    NOISE_DENSITY = 0.15  
    MEAN_NOISE_SPAN_LENGTH = 3.0 
    
    # Monitoring 
    REPORT_TO = 'wandb' if WANDB_AVAILABLE else 'tensorboard' 
    WANDB_PROJECT = 'mt5-ja-ne-pretraining'
    WANDB_RUN_NAME = f'mt5-pretrain-{datetime.now().strftime("%Y%m%d-%H%M%S")}'
    
    # Miscellaneous 
    SEED = 42
    RESUME_FROM_CHECKPOINT = True 
    PUSH_TO_HUB = False 
    HUB_MODEL_ID = 'your-username/mt5-ja-ne-pretrained'

    @classmethod
    def to_dict(cls) -> Dict[str, Any]:
        """Convert config to dictionary"""
        return {
            key: value for key, value in cls.__dict__.items()
            if not key.startswith('_') and not callable(value)
        }
    
    @classmethod
    def print_config(cls):
        """Pretty print configuration"""
        print("\n" + "=" * 80)
        print("📋 PRETRAINING CONFIGURATION")
        print("=" * 80)
        
        sections = {
            'Paths': ['DATASET_FILES', 'TOKENIZER_PATH', 'OUTPUT_DIR'],
            'Model': ['MODEL_NAME', 'TRAIN_FROM_SCRATCH', 'D_MODEL', 'NUM_LAYERS'],
            'Data': ['MAX_LENGTH', 'VALIDATION_SPLIT', 'STREAMING'],
            'Training': ['BATCH_SIZE', 'GRADIENT_ACCUMULATION_STEPS', 'LEARNING_RATE', 'NUM_TRAIN_EPOCHS'],
            'Optimization': ['FP16', 'GRADIENT_CHECKPOINTING', 'OPTIM'],
            'T5 Specific': ['NOISE_DENSITY', 'MEAN_NOISE_SPAN_LENGTH'],
        }
        
        for section, keys in sections.items():
            print(f"\n{section}:")
            for key in keys:
                value = getattr(cls, key, 'N/A')
                if isinstance(value, list) and len(value) > 3:
                    value = f"[{len(value)} files]"
                print(f"  {key:.<45} {value}")
        
        print("\n" + "=" * 80 + "\n")


# Set random seed for reproducibility
set_seed(PretrainingConfig.SEED)

# Print configuration
PretrainingConfig.print_config()

# Create directories
os.makedirs(PretrainingConfig.OUTPUT_DIR, exist_ok=True)
os.makedirs(PretrainingConfig.LOGGING_DIR, exist_ok=True)
os.makedirs(PretrainingConfig.FINAL_MODEL_DIR, exist_ok=True)

print("Configuration complete!")

In [ ]:
def load_json_datasets(
    file_paths: List[str],
    streaming: bool = False,
    validation_split: float = 0.05,
    seed: int = 42
) -> DatasetDict:

    print("=" * 80)
    print(" LOADING DATASETS")
    print("=" * 80)
    print(f"\n Loading {len(file_paths)} JSON files...\n")
    
    # Verify files exist
    for i, file_path in enumerate(file_paths, 1):
        if not os.path.exists(file_path):
            raise FileNotFoundError(f" Dataset file not found: {file_path}")
        
        # Get file size
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  {i}. ✓ {file_path} ({file_size_mb:.2f} MB)")
    
    # Load datasets
    print(f"\n Loading datasets...")
    dataset = load_dataset(
        'json',
        data_files={'train': file_paths},
        streaming=streaming
    )
    
    # Print dataset info
    if not streaming:
        print(f"\n Dataset Statistics:")
        print(f"  Total examples: {len(dataset['train']):,}")
        
        # Sample text
        sample_text = dataset['train'][0]['text']
        print(f"\n Sample text (first 300 chars):")
        print(f"  {sample_text[:300]}...")
        
        # Text length statistics
        text_lengths = [len(ex['text']) for ex in dataset['train'].select(range(min(1000, len(dataset['train']))))]
        print(f"\n Text length statistics (first 1000 examples):")
        print(f"  Min: {min(text_lengths):,} chars")
        print(f"  Max: {max(text_lengths):,} chars")
        print(f"  Mean: {np.mean(text_lengths):.0f} chars")
        print(f"  Median: {np.median(text_lengths):.0f} chars")
        
        # Split into train/validation
        print(f"\n Splitting dataset (validation={validation_split*100:.1f}%)...")
        dataset = dataset['train'].train_test_split(
            test_size=validation_split,
            seed=seed
        )
        
        # Rename 'test' to 'validation' for clarity
        dataset = DatasetDict({
            'train': dataset['train'],
            'validation': dataset['test']
        })
        
        print(f"\n Dataset split complete:")
        print(f"  Train examples: {len(dataset['train']):,}")
        print(f"  Validation examples: {len(dataset['validation']):,}")
    else:
        print(f"\n Streaming dataset loaded (example count unavailable in streaming mode)")
    
    print("\n" + "=" * 80)
    print(" DATASET LOADING COMPLETE")
    print("=" * 80 + "\n")
    
    return dataset


# Load datasets
dataset = load_json_datasets(
    file_paths=PretrainingConfig.DATASET_FILES,
    streaming=PretrainingConfig.STREAMING,
    validation_split=PretrainingConfig.VALIDATION_SPLIT,
    seed=PretrainingConfig.SEED
)

: 

In [ ]:
print("=" * 80)
print(" TOKENIZER LOADING")
print("=" * 80)

# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained(
    PretrainingConfig.TOKENIZER_PATH,
    model_max_length=PretrainingConfig.MAX_LENGTH
)

print(f"\n Tokenizer loaded: {PretrainingConfig.TOKENIZER_PATH}")
print(f"  Vocabulary size: {len(tokenizer):,}")
print(f"  Model max length: {tokenizer.model_max_length}")
print(f"  PAD token: '{tokenizer.pad_token}' (ID: {tokenizer.pad_token_id})")
print(f"  EOS token: '{tokenizer.eos_token}' (ID: {tokenizer.eos_token_id})")
print(f"  UNK token: '{tokenizer.unk_token}' (ID: {tokenizer.unk_token_id})")

# Test tokenization
sample_text = dataset['train'][0]['text'][:100]
tokens = tokenizer(sample_text)
print(f"\n Sample tokenization:")
print(f"  Input: {sample_text}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(tokens['input_ids'][:20])}...")
print(f"  Token IDs: {tokens['input_ids'][:20]}...")

print("\n" + "=" * 80)
print(" TOKENIZING DATASET")
print("=" * 80)

def preprocess_function(examples):
    """
    Tokenize the 'text' field from JSON files
    """
    return tokenizer(
        examples['text'],
        max_length=PretrainingConfig.MAX_LENGTH,
        truncation=True,
        padding=False,  # We'll use dynamic padding in data collator
        return_attention_mask=True
    )

# Tokenize datasets
print("\n🔄 Tokenizing train split...")
tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=1000,
    num_proc=PretrainingConfig.NUM_PROC,
    remove_columns=['text'],  # Remove original text after tokenization
    desc="Tokenizing"
)

print("\n✅ Tokenization complete!")
print(f"\n📊 Tokenized Dataset:")
print(tokenized_datasets)

# Show sample
print(f"\n📝 Sample tokenized example:")
sample = tokenized_datasets['train'][0]
print(f"  Input IDs shape: {len(sample['input_ids'])}")
print(f"  Attention mask shape: {len(sample['attention_mask'])}")
print(f"  First 20 tokens: {sample['input_ids'][:20]}")

print("\n" + "=" * 80 + "\n")

In [ ]:
@dataclass
class DataCollatorForT5MLM:
    
    tokenizer: T5Tokenizer
    noise_density: float = 0.15
    mean_noise_span_length: float = 3.0
    input_length: int = 512
    target_length: int = 114  # Typically ~max_length/4.5
    pad_token_id: int = 0
    decoder_start_token_id: int = 0
    
    def __call__(self, examples: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
        
        # Get input_ids from examples
        input_ids = [example['input_ids'] for example in examples]
        
        batch_input_ids = []
        batch_labels = []
        
        for ids in input_ids:
            # Convert to numpy array
            ids = np.array(ids)
            
            # Apply span corruption
            corrupted_ids, labels = self.create_sentinel_noise(
                ids,
                self.noise_density,
                self.mean_noise_span_length
            )
            
            batch_input_ids.append(corrupted_ids)
            batch_labels.append(labels)
        
        # Pad to max length
        batch = self.pad_sequences(
            input_ids=batch_input_ids,
            labels=batch_labels
        )
        
        return batch
    
    def create_sentinel_noise(
        self,
        input_ids: np.ndarray,
        noise_density: float,
        mean_noise_span_length: float
    ) -> tuple:
       
        length = len(input_ids)
        
        # Calculate number of tokens to corrupt
        num_noise_tokens = int(np.round(length * noise_density))
        num_noise_tokens = min(max(num_noise_tokens, 1), length - 1)
        
        # Calculate number of noise spans
        num_noise_spans = int(np.round(num_noise_tokens / mean_noise_span_length))
        num_noise_spans = max(num_noise_spans, 1)
        
        # Create noise mask
        noise_mask = np.zeros(length, dtype=bool)
        
        # Randomly select span starts
        if length > mean_noise_span_length:
            span_starts = np.random.choice(
                length - int(mean_noise_span_length),
                size=min(num_noise_spans, length // 2),
                replace=False
            )
            
            # Mark spans as noise
            for start in span_starts:
                span_length = min(
                    int(np.random.poisson(mean_noise_span_length)),
                    length - start
                )
                noise_mask[start:start + span_length] = True
        
        # Get sentinel tokens (special tokens at end of vocabulary)
        sentinel_ids = list(range(
            self.tokenizer.vocab_size - 100,
            self.tokenizer.vocab_size
        ))
        
        # Build corrupted input and labels
        input_ids_corrupted = []
        labels = []
        
        sentinel_idx = 0
        i = 0
        
        while i < length:
            if noise_mask[i]:
                # Start of a corrupted span
                if sentinel_idx < len(sentinel_ids):
                    sentinel = sentinel_ids[sentinel_idx]
                    
                    # Add sentinel to input
                    input_ids_corrupted.append(sentinel)
                    
                    # Add sentinel to labels
                    labels.append(sentinel)
                    
                    # Add corrupted tokens to labels
                    while i < length and noise_mask[i]:
                        labels.append(int(input_ids[i]))
                        i += 1
                    
                    sentinel_idx += 1
                else:
                    i += 1
            else:
                # Non-corrupted token
                input_ids_corrupted.append(int(input_ids[i]))
                i += 1
        
        # Add final sentinel to labels
        if sentinel_idx < len(sentinel_ids):
            labels.append(sentinel_ids[sentinel_idx])
        
        return input_ids_corrupted, labels
    
    def pad_sequences(
        self,
        input_ids: List[List[int]],
        labels: List[List[int]]
    ) -> Dict[str, torch.Tensor]:
      
        # Find max lengths
        max_input_length = min(max(len(ids) for ids in input_ids), self.input_length)
        max_label_length = min(max(len(lab) for lab in labels), self.target_length)
        
        # Pad inputs
        padded_input_ids = []
        attention_mask = []
        
        for ids in input_ids:
            # Truncate if necessary
            ids = ids[:max_input_length]
            
            padding_length = max_input_length - len(ids)
            padded_ids = ids + [self.pad_token_id] * padding_length
            mask = [1] * len(ids) + [0] * padding_length
            
            padded_input_ids.append(padded_ids)
            attention_mask.append(mask)
        
        # Pad labels
        padded_labels = []
        
        for lab in labels:
            # Truncate if necessary
            lab = lab[:max_label_length]
            
            padding_length = max_label_length - len(lab)
            padded = lab + [-100] * padding_length  # -100 is ignored in loss
            
            padded_labels.append(padded)
        
        # Convert to tensors
        batch = {
            'input_ids': torch.tensor(padded_input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(padded_labels, dtype=torch.long)
        }
        
        return batch


# Initialize data collator
data_collator = DataCollatorForT5MLM(
    tokenizer=tokenizer,
    noise_density=PretrainingConfig.NOISE_DENSITY,
    mean_noise_span_length=PretrainingConfig.MEAN_NOISE_SPAN_LENGTH,
    input_length=PretrainingConfig.MAX_LENGTH,
    target_length=int(PretrainingConfig.MAX_LENGTH / 4.5),
    pad_token_id=tokenizer.pad_token_id
)

print("✅ Data collator initialized!")
print(f"  Noise density: {PretrainingConfig.NOISE_DENSITY}")
print(f"  Mean noise span length: {PretrainingConfig.MEAN_NOISE_SPAN_LENGTH}")

# Test data collator
print("\n🧪 Testing data collator...")
sample_batch = [tokenized_datasets['train'][i] for i in range(2)]
collated_batch = data_collator(sample_batch)

print(f"\n📊 Collated batch shapes:")
for key, value in collated_batch.items():
    print(f"  {key}: {value.shape}")

print(f"\n📝 Sample corrupted input (first 30 tokens):")
print(f"  {tokenizer.convert_ids_to_tokens(collated_batch['input_ids'][0][:30].tolist())}")

print(f"\n🎯 Sample target (first 30 tokens):")
label_tokens = collated_batch['labels'][0][:30]
label_tokens = [t.item() for t in label_tokens if t != -100]
print(f"  {tokenizer.convert_ids_to_tokens(label_tokens[:30])}")

print("\n✅ Data collator test complete!\n")

In [ ]:
print("=" * 80)
print("🤖 MODEL INITIALIZATION")
print("=" * 80)

if PretrainingConfig.TRAIN_FROM_SCRATCH:
    # Initialize model from scratch
    print("\n🏗️  Initializing model from scratch...\n")
    
    config = T5Config(
        vocab_size=PretrainingConfig.VOCAB_SIZE,
        d_model=PretrainingConfig.D_MODEL,
        d_ff=PretrainingConfig.D_FF,
        num_layers=PretrainingConfig.NUM_LAYERS,
        num_decoder_layers=PretrainingConfig.NUM_DECODER_LAYERS,
        num_heads=PretrainingConfig.NUM_HEADS,
        relative_attention_num_buckets=32,
        relative_attention_max_distance=128,
        dropout_rate=PretrainingConfig.DROPOUT_RATE,
        layer_norm_epsilon=1e-6,
        initializer_factor=1.0,
        feed_forward_proj='gated-gelu',
        use_cache=False,
    )
    
    model = T5ForConditionalGeneration(config)
    print("✅ Model initialized from scratch")
else:
    # Load pretrained model
    print(f"\n📥 Loading pretrained model: {PretrainingConfig.MODEL_NAME}\n")
    
    model = T5ForConditionalGeneration.from_pretrained(
        PretrainingConfig.MODEL_NAME
    )
    print(f"✅ Pretrained model loaded: {PretrainingConfig.MODEL_NAME}")

# Resize token embeddings if vocabulary changed
model.resize_token_embeddings(len(tokenizer))

# Enable gradient checkpointing for memory efficiency
if PretrainingConfig.GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    print("✅ Gradient checkpointing enabled")

# Model info
print(f"\n📊 Model Information:")
print(f"  Model type: {model.config.model_type}")
print(f"  Hidden size (d_model): {model.config.d_model}")
print(f"  Number of layers: {model.config.num_layers}")
print(f"  Number of heads: {model.config.num_heads}")
print(f"  Vocabulary size: {model.config.vocab_size:,}")
print(f"  Feed-forward size: {model.config.d_ff:,}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n🎯 Parameters:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e9:.2f} GB (FP32)")
if PretrainingConfig.FP16:
    print(f"  Model size: ~{total_params * 2 / 1e9:.2f} GB (FP16)")

print("\n" + "=" * 80 + "\n")

In [ ]:
print("=" * 80)
print("⚙️  TRAINING ARGUMENTS")
print("=" * 80)

training_args = TrainingArguments(
    # Output
    output_dir=PretrainingConfig.OUTPUT_DIR,
    overwrite_output_dir=True,
    
    # Training hyperparameters
    num_train_epochs=PretrainingConfig.NUM_TRAIN_EPOCHS,
    max_steps=PretrainingConfig.MAX_STEPS,
    per_device_train_batch_size=PretrainingConfig.BATCH_SIZE,
    per_device_eval_batch_size=PretrainingConfig.BATCH_SIZE,
    gradient_accumulation_steps=PretrainingConfig.GRADIENT_ACCUMULATION_STEPS,
    
    # Optimization
    learning_rate=PretrainingConfig.LEARNING_RATE,
    warmup_steps=PretrainingConfig.WARMUP_STEPS,
    weight_decay=PretrainingConfig.WEIGHT_DECAY,
    max_grad_norm=PretrainingConfig.MAX_GRAD_NORM,
    adam_epsilon=PretrainingConfig.ADAM_EPSILON,
    adam_beta1=PretrainingConfig.ADAM_BETA1,
    adam_beta2=PretrainingConfig.ADAM_BETA2,
    lr_scheduler_type=PretrainingConfig.LR_SCHEDULER_TYPE,
    optim=PretrainingConfig.OPTIM,
    
    # Precision & Performance
    fp16=PretrainingConfig.FP16,
    bf16=PretrainingConfig.BF16,
    fp16_full_eval=PretrainingConfig.FP16,
    dataloader_num_workers=PretrainingConfig.DATALOADER_NUM_WORKERS,
    dataloader_pin_memory=PretrainingConfig.DATALOADER_PIN_MEMORY,
    group_by_length=False,  # Not useful for T5
    
    # Checkpointing
    save_strategy=PretrainingConfig.SAVE_STRATEGY,
    save_steps=PretrainingConfig.SAVE_STEPS,
    save_total_limit=PretrainingConfig.SAVE_TOTAL_LIMIT,
    save_safetensors=True,
    
    # Evaluation
    evaluation_strategy=PretrainingConfig.EVAL_STRATEGY,
    eval_steps=PretrainingConfig.EVAL_STEPS,
    eval_accumulation_steps=None,
    
    # Logging
    logging_dir=PretrainingConfig.LOGGING_DIR,
    logging_strategy='steps',
    logging_steps=PretrainingConfig.LOGGING_STEPS,
    logging_first_step=True,
    report_to=PretrainingConfig.REPORT_TO,
    
    # Other
    seed=PretrainingConfig.SEED,
    data_seed=PretrainingConfig.SEED,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    ignore_data_skip=False,
    
    # Hub
    push_to_hub=PretrainingConfig.PUSH_TO_HUB,
    hub_model_id=PretrainingConfig.HUB_MODEL_ID if PretrainingConfig.PUSH_TO_HUB else None,
    hub_strategy='every_save' if PretrainingConfig.PUSH_TO_HUB else None,
    
    # Distributed training
    ddp_find_unused_parameters=False,
)

print("\n✅ Training arguments configured!")
print(f"\n📊 Key Settings:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size (per device): {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * max(1, torch.cuda.device_count())}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Warmup steps: {training_args.warmup_steps}")
print(f"  FP16: {training_args.fp16}")
print(f"  Save steps: {training_args.save_steps}")
print(f"  Eval steps: {training_args.eval_steps}")
print(f"  Logging: {training_args.report_to}")

print("\n" + "=" * 80 + "\n")

In [ ]:
class PerplexityCallback(TrainerCallback):
    """
    Callback to compute and log perplexity during training
    """
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            # Add perplexity to logs
            if 'loss' in logs:
                logs['train_perplexity'] = np.exp(logs['loss'])
            
            if 'eval_loss' in logs:
                logs['eval_perplexity'] = np.exp(logs['eval_loss'])


class DetailedLoggingCallback(TrainerCallback):
    """
    Callback for detailed logging of training progress
    """
    
    def on_step_end(self, args, state, control, **kwargs):
        # Log every N steps
        if state.global_step % (args.logging_steps * 10) == 0:
            print(f"\n{'='*60}")
            print(f"Step {state.global_step}/{state.max_steps if state.max_steps > 0 else 'N/A'}")
            print(f"Epoch: {state.epoch:.2f}")
            if state.log_history:
                latest_log = state.log_history[-1]
                if 'loss' in latest_log:
                    print(f"Loss: {latest_log['loss']:.4f}")
                if 'learning_rate' in latest_log:
                    print(f"Learning Rate: {latest_log['learning_rate']:.2e}")
            print(f"{'='*60}\n")


class MemoryMonitorCallback(TrainerCallback):
    """
    Monitor GPU memory usage during training
    """
    
    def on_step_end(self, args, state, control, **kwargs):
        if torch.cuda.is_available() and state.global_step % 1000 == 0:
            for i in range(torch.cuda.device_count()):
                mem_allocated = torch.cuda.memory_allocated(i) / 1e9
                mem_reserved = torch.cuda.memory_reserved(i) / 1e9
                print(f"GPU {i} Memory: {mem_allocated:.2f}GB allocated, {mem_reserved:.2f}GB reserved")


class CheckpointCallback(TrainerCallback):
    """
    Custom checkpoint callback for saving additional artifacts
    """
    
    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = f"{args.output_dir}/checkpoint-{state.global_step}"
        
        # Save training state metadata
        metadata = {
            'global_step': state.global_step,
            'epoch': state.epoch,
            'best_metric': state.best_metric,
            'best_model_checkpoint': state.best_model_checkpoint,
            'timestamp': datetime.now().isoformat(),
        }
        
        metadata_path = f"{checkpoint_dir}/training_metadata.json"
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"\n💾 Checkpoint saved: {checkpoint_dir}")
        print(f"   Step: {state.global_step}, Epoch: {state.epoch:.2f}")
        if state.best_metric is not None:
            print(f"   Best eval loss: {state.best_metric:.4f}\n")


# Initialize callbacks
callbacks = [
    PerplexityCallback(),
    DetailedLoggingCallback(),
    CheckpointCallback(),
]

if torch.cuda.is_available():
    callbacks.append(MemoryMonitorCallback())

# Add early stopping (optional)
# callbacks.append(EarlyStoppingCallback(early_stopping_patience=3))

print("✅ Callbacks initialized!")
print(f"  Active callbacks: {len(callbacks)}")
for callback in callbacks:
    print(f"    - {callback.__class__.__name__}")
print()

In [ ]:
print("=" * 80)
print("🎯 TRAINER INITIALIZATION")
print("=" * 80)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=callbacks,
)

print("\n✅ Trainer initialized successfully!")

# Print training overview
print(f"\n📊 Training Overview:")
print(f"  Train samples: {len(tokenized_datasets['train']):,}")
print(f"  Validation samples: {len(tokenized_datasets['validation']):,}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")

# Calculate total steps
total_steps = len(tokenized_datasets['train']) // (
    training_args.per_device_train_batch_size * 
    training_args.gradient_accumulation_steps *
    max(1, torch.cuda.device_count())
) * training_args.num_train_epochs

print(f"  Total training steps: ~{total_steps:,}")
print(f"  Warmup steps: {training_args.warmup_steps:,}")
print(f"  Save every: {training_args.save_steps:,} steps")
print(f"  Evaluate every: {training_args.eval_steps:,} steps")

# Estimated training time (very rough estimate)
if torch.cuda.is_available():
    # Rough estimate: ~1-2 seconds per step on modern GPU
    estimated_hours = (total_steps * 1.5) / 3600
    print(f"\n⏱️  Estimated training time: ~{estimated_hours:.1f} hours")
    print(f"   (This is a rough estimate and may vary significantly)")

print("\n" + "=" * 80 + "\n")

In [ ]:
print("=" * 80)
print("🚀 STARTING TRAINING")
print("=" * 80)

# Check for existing checkpoints
checkpoint = None
if PretrainingConfig.RESUME_FROM_CHECKPOINT:
    if os.path.exists(PretrainingConfig.OUTPUT_DIR):
        checkpoints = [
            d for d in os.listdir(PretrainingConfig.OUTPUT_DIR)
            if d.startswith('checkpoint-')
        ]
        if checkpoints:
            # Get latest checkpoint
            checkpoint = os.path.join(
                PretrainingConfig.OUTPUT_DIR,
                sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
            )
            print(f"\n🔄 Resuming from checkpoint: {checkpoint}\n")
        else:
            print("\n✨ No checkpoints found. Starting fresh training.\n")
    else:
        print("\n✨ Starting fresh training.\n")
else:
    print("\n✨ Starting fresh training (auto-resume disabled).\n")

# Initialize W&B if available
if WANDB_AVAILABLE and PretrainingConfig.REPORT_TO == 'wandb':
    try:
        wandb.init(
            project=PretrainingConfig.WANDB_PROJECT,
            name=PretrainingConfig.WANDB_RUN_NAME,
            config=PretrainingConfig.to_dict(),
            resume='allow' if checkpoint else False
        )
        print("✅ W&B initialized")
    except Exception as e:
        print(f"⚠️  W&B initialization failed: {e}")

print("\n" + "=" * 80)
print("⚡ TRAINING IN PROGRESS")
print("=" * 80)
print("\nPress Ctrl+C to stop training (checkpoints will be saved)\n")

# Start training
try:
    train_result = trainer.train(resume_from_checkpoint=checkpoint)
    
    print("\n" + "=" * 80)
    print("✅ TRAINING COMPLETED SUCCESSFULLY")
    print("=" * 80)
    
    # Print training metrics
    print("\n📊 Final Training Metrics:")
    for key, value in train_result.metrics.items():
        print(f"  {key}: {value}")
    
    # Save metrics
    metrics_file = f"{PretrainingConfig.OUTPUT_DIR}/train_results.json"
    trainer.save_metrics("train", train_result.metrics)
    print(f"\n💾 Metrics saved to: {metrics_file}")
    
except KeyboardInterrupt:
    print("\n" + "=" * 80)
    print("⚠️  TRAINING INTERRUPTED")
    print("=" * 80)
    print("\n💾 Checkpoint saved. You can resume training later.\n")
    
except Exception as e:
    print("\n" + "=" * 80)
    print("❌ TRAINING FAILED")
    print("=" * 80)
    print(f"\nError: {str(e)}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "=" * 80 + "\n")

In [ ]:
print("=" * 80)
print("📊 MODEL EVALUATION")
print("=" * 80)

# Evaluate on validation set
print("\n🔍 Evaluating on validation set...\n")
eval_results = trainer.evaluate()

print("\n✅ Evaluation complete!\n")
print("📊 Validation Metrics:")
for key, value in eval_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")

# Calculate perplexity
if 'eval_loss' in eval_results:
    perplexity = np.exp(eval_results['eval_loss'])
    print(f"\n🎯 Perplexity: {perplexity:.2f}")

# Save evaluation results
eval_file = f"{PretrainingConfig.OUTPUT_DIR}/eval_results.json"
trainer.save_metrics("eval", eval_results)
print(f"\n💾 Evaluation results saved to: {eval_file}")

print("\n" + "=" * 80 + "\n")

In [ ]:
print("=" * 80)
print("💾 MEMORY OPTIMIZATION STATUS")
print("=" * 80)

print("\n📊 Optimization Settings:")
print(f"  FP16: {'✅ Enabled' if PretrainingConfig.FP16 else '❌ Disabled'}")
print(f"  BF16: {'✅ Enabled' if PretrainingConfig.BF16 else '❌ Disabled'}")
print(f"  Gradient Checkpointing: {'✅ Enabled' if PretrainingConfig.GRADIENT_CHECKPOINTING else '❌ Disabled'}")
print(f"  Gradient Accumulation: {PretrainingConfig.GRADIENT_ACCUMULATION_STEPS} steps")

if torch.cuda.is_available():
    print("\n🎮 GPU Memory Usage:")
    for i in range(torch.cuda.device_count()):
        mem_allocated = torch.cuda.memory_allocated(i) / 1e9
        mem_reserved = torch.cuda.memory_reserved(i) / 1e9
        mem_total = torch.cuda.get_device_properties(i).total_memory / 1e9
        
        print(f"\n  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Allocated: {mem_allocated:.2f}GB / {mem_total:.2f}GB ({mem_allocated/mem_total*100:.1f}%)")
        print(f"    Reserved:  {mem_reserved:.2f}GB / {mem_total:.2f}GB ({mem_reserved/mem_total*100:.1f}%)")
    
    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()
    print("\n  ✅ GPU cache cleared")
else:
    print("\n⚠️  No GPU detected")

print("\n" + "=" * 80 + "\n")

In [ ]:
print("=" * 80)
print("💾 SAVING FINAL MODEL")
print("=" * 80)

print(f"\n📁 Saving to: {PretrainingConfig.FINAL_MODEL_DIR}\n")

# Save model
trainer.save_model(PretrainingConfig.FINAL_MODEL_DIR)
print("✅ Model saved")

# Save tokenizer
tokenizer.save_pretrained(PretrainingConfig.FINAL_MODEL_DIR)
print("✅ Tokenizer saved")

# Save training arguments
training_args_file = f"{PretrainingConfig.FINAL_MODEL_DIR}/training_args.json"
with open(training_args_file, 'w') as f:
    json.dump(training_args.to_dict(), f, indent=2)
print("✅ Training arguments saved")

# Save configuration
config_file = f"{PretrainingConfig.FINAL_MODEL_DIR}/pretraining_config.json"
with open(config_file, 'w') as f:
    json.dump(PretrainingConfig.to_dict(), f, indent=2)
print("✅ Pretraining configuration saved")

# Create README
readme_content = f"""# mT5 Japanese-Nepali Pretrained Model

## Model Details
- Base model: {PretrainingConfig.MODEL_NAME}
- Training date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
- Epochs: {PretrainingConfig.NUM_TRAIN_EPOCHS}
- Training samples: {len(tokenized_datasets['train']):,}
- Validation samples: {len(tokenized_datasets['validation']):,}

## Training Configuration
- Batch size: {PretrainingConfig.BATCH_SIZE}
- Learning rate: {PretrainingConfig.LEARNING_RATE}
- Max length: {PretrainingConfig.MAX_LENGTH}
- Noise density: {PretrainingConfig.NOISE_DENSITY}
- Mean noise span: {PretrainingConfig.MEAN_NOISE_SPAN_LENGTH}

## Usage
```python
from transformers import T5ForConditionalGeneration, T5Tokenizer

model = T5ForConditionalGeneration.from_pretrained('{PretrainingConfig.FINAL_MODEL_DIR}')
tokenizer = T5Tokenizer.from_pretrained('{PretrainingConfig.FINAL_MODEL_DIR}')
```
"""

readme_file = f"{PretrainingConfig.FINAL_MODEL_DIR}/README.md"
with open(readme_file, 'w') as f:
    f.write(readme_content)
print("✅ README.md created")

print(f"\n📦 Model package contents:")
for item in os.listdir(PretrainingConfig.FINAL_MODEL_DIR):
    item_path = os.path.join(PretrainingConfig.FINAL_MODEL_DIR, item)
    if os.path.isfile(item_path):
        size_mb = os.path.getsize(item_path) / (1024 * 1024)
        print(f"  - {item} ({size_mb:.2f} MB)")
    else:
        print(f"  - {item}/ (directory)")

# Push to Hub (if enabled)
if PretrainingConfig.PUSH_TO_HUB:
    print("\n🚀 Pushing to Hugging Face Hub...")
    try:
        trainer.push_to_hub(
            commit_message=f"Upload mT5 pretrained model - {datetime.now().strftime('%Y-%m-%d')}"
        )
        print(f"✅ Model pushed to: https://huggingface.co/{PretrainingConfig.HUB_MODEL_ID}")
    except Exception as e:
        print(f"❌ Failed to push to Hub: {e}")

print("\n" + "=" * 80)
print("✅ MODEL SAVING COMPLETE")
print("=" * 80 + "\n")

In [ ]:
print("=" * 80)
print("📈 TRAINING VISUALIZATION")
print("=" * 80)

# Load training history
history_df = pd.DataFrame(trainer.state.log_history)

print(f"\n📊 Training history loaded: {len(history_df)} log entries\n")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('mT5 Pretraining Progress', fontsize=16, fontweight='bold')

# 1. Training Loss
train_loss = history_df[history_df['loss'].notna()]
if len(train_loss) > 0:
    axes[0, 0].plot(train_loss['step'], train_loss['loss'], label='Train Loss', linewidth=2)
    axes[0, 0].set_xlabel('Steps', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

# 2. Validation Loss
eval_loss = history_df[history_df['eval_loss'].notna()]
if len(eval_loss) > 0:
    axes[0, 1].plot(eval_loss['step'], eval_loss['eval_loss'], 
                    label='Validation Loss', color='orange', linewidth=2)
    axes[0, 1].set_xlabel('Steps', fontsize=12)
    axes[0, 1].set_ylabel('Loss', fontsize=12)
    axes[0, 1].set_title('Validation Loss', fontsize=14, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

# 3. Learning Rate Schedule
lr_data = history_df[history_df['learning_rate'].notna()]
if len(lr_data) > 0:
    axes[1, 0].plot(lr_data['step'], lr_data['learning_rate'], 
                    label='Learning Rate', color='green', linewidth=2)
    axes[1, 0].set_xlabel('Steps', fontsize=12)
    axes[1, 0].set_ylabel('Learning Rate', fontsize=12)
    axes[1, 0].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))

# 4. Combined Train/Val Loss
if len(train_loss) > 0:
    axes[1, 1].plot(train_loss['step'], train_loss['loss'], 
                    label='Train Loss', linewidth=2, alpha=0.7)
if len(eval_loss) > 0:
    axes[1, 1].plot(eval_loss['step'], eval_loss['eval_loss'], 
                    label='Validation Loss', linewidth=2, alpha=0.7)
axes[1, 1].set_xlabel('Steps', fontsize=12)
axes[1, 1].set_ylabel('Loss', fontsize=12)
axes[1, 1].set_title('Train vs Validation Loss', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PretrainingConfig.OUTPUT_DIR}/training_progress.png", dpi=300, bbox_inches='tight')
print(f"✅ Training progress plot saved to: {PretrainingConfig.OUTPUT_DIR}/training_progress.png")
plt.show()

# Print summary statistics
print("\n" + "=" * 80)
print("📊 TRAINING SUMMARY")
print("=" * 80)

if len(train_loss) > 0:
    print(f"\n🏋️ Training Loss:")
    print(f"  Initial: {train_loss['loss'].iloc[0]:.4f}")
    print(f"  Final: {train_loss['loss'].iloc[-1]:.4f}")
    print(f"  Best: {train_loss['loss'].min():.4f}")
    print(f"  Improvement: {(1 - train_loss['loss'].iloc[-1] / train_loss['loss'].iloc[0]) * 100:.2f}%")

if len(eval_loss) > 0:
    print(f"\n📊 Validation Loss:")
    print(f"  Initial: {eval_loss['eval_loss'].iloc[0]:.4f}")
    print(f"  Final: {eval_loss['eval_loss'].iloc[-1]:.4f}")
    print(f"  Best: {eval_loss['eval_loss'].min():.4f}")
    print(f"  Improvement: {(1 - eval_loss['eval_loss'].iloc[-1] / eval_loss['eval_loss'].iloc[0]) * 100:.2f}%")
    
    # Perplexity
    print(f"\n🎯 Perplexity:")
    print(f"  Initial: {np.exp(eval_loss['eval_loss'].iloc[0]):.2f}")
    print(f"  Final: {np.exp(eval_loss['eval_loss'].iloc[-1]):.2f}")
    print(f"  Best: {np.exp(eval_loss['eval_loss'].min()):.2f}")

print("\n" + "=" * 80)

# Save history to CSV
history_file = f"{PretrainingConfig.OUTPUT_DIR}/training_history.csv"
history_df.to_csv(history_file, index=False)
print(f"\n💾 Training history saved to: {history_file}")

print("\n" + "=" * 80)
print("✅ VISUALIZATION COMPLETE")
print("=" * 80 + "\n")